#### 1.Carga de datos

In [70]:
import pandas as pd

In [71]:
df = pd.read_parquet("../data/processed/events_clean.parquet")
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00+00:00,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00+00:00,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01+00:00,view,17302664,2053013553853497655,unknown,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01+00:00,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01+00:00,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


#### 2.Preparación de los datos

In [72]:
df_purchase = df[df["event_type"] == "purchase"].copy()
df_purchase = df_purchase.sort_values(["user_id", "event_time"])
df_purchase.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
96795,2019-11-01 03:49:22+00:00,purchase,22600026,2165087460176953468,auto.accessories.compressor,unknown,33.45,356520186,3e3403b6-2a19-4a1d-aef2-ee4dce0dc943
483671,2019-11-01 09:05:50+00:00,purchase,22700084,2053013556168753601,unknown,force,244.28,397023870,85566778-28dd-439e-958f-a5c972133c3e
5493,2019-11-01 00:34:18+00:00,purchase,1004768,2053013555631882655,electronics.smartphone,samsung,242.72,486999716,2666d6ed-08d8-4a83-b487-2cccbfbf34ff
215379,2019-11-01 05:39:29+00:00,purchase,13300618,2053013557166998015,unknown,askona,200.52,509881222,3164adb2-d1d2-45c7-898e-452e10e5b557
966776,2019-11-01 15:12:55+00:00,purchase,1801690,2053013554415534427,electronics.video.tv,samsung,369.45,512363923,6d60dc8b-c8aa-4b5b-9dad-89ea9d3bbdd0


#### 3.Definición de churn

In [73]:
last_purchase = df_purchase.groupby("user_id")["event_time"].max()
last_purchase.head()

user_id
356520186   2019-11-01 03:49:22+00:00
397023870   2019-11-01 09:05:50+00:00
486999716   2019-11-01 00:34:18+00:00
509881222   2019-11-01 05:39:29+00:00
512363923   2019-11-01 15:12:55+00:00
Name: event_time, dtype: datetime64[ns, UTC]

In [74]:
cutoff_date = df["event_time"].max() - pd.Timedelta(days=7)
cutoff_date

Timestamp('2019-10-25 15:32:10+0000', tz='UTC')

In [75]:
churn_df = last_purchase.reset_index()
churn_df["churn"] = churn_df["event_time"] < cutoff_date
churn_df.head()

,user_id,event_time,churn
0,356520186,2019-11-01 03:49:22+00:00,False
1,397023870,2019-11-01 09:05:50+00:00,False
2,486999716,2019-11-01 00:34:18+00:00,False
3,509881222,2019-11-01 05:39:29+00:00,False
4,512363923,2019-11-01 15:12:55+00:00,False


#### 4.Cálculo del churn rate

In [76]:
churn_rate = churn_df["churn"].mean()
churn_rate

np.float64(0.0)

In [77]:
print(f"Churn rate: {churn_rate:.2%}")

Churn rate: 0.00%


#### 5.Distribución de churn

In [78]:
churn_df["churn"].value_counts(normalize=True)

churn
False    1.0
Name: proportion, dtype: float64

#### 6. Definición alternativa de churn (dentro del período)

In [79]:
purchase_counts = df_purchase.groupby("user_id").size()
churn_df = purchase_counts.reset_index(name="purchase_count")
churn_df["churn"] = churn_df["purchase_count"] == 1
churn_df.head()

,user_id,purchase_count,churn
0,356520186,1,True
1,397023870,1,True
2,486999716,1,True
3,509881222,1,True
4,512363923,1,True


In [80]:
churn_rate = churn_df["churn"].mean()
print(f"Churn rate: {churn_rate:.2%}")

Churn rate: 80.08%


#### 7.Churn por categoría

In [81]:
df_purchase = df[df["event_type"] == "purchase"].copy()
df_purchase = df_purchase[df_purchase["category_code"] != "unknown"]

#### 8.Cálculo de churn por categoría

In [82]:
churn_category = (
    df_purchase
    .groupby(["category_code", "user_id"])
    .size()
    .reset_index(name="purchase_counts")
)

churn_category["churn"] = churn_category["purchase_counts"] == 1

churn_category_rate = (
    churn_category
    .groupby("category_code")["churn"]
    .mean()
    .reset_index()
    .sort_values(by="churn", ascending=False)
)

churn_category_rate.head(10)

,category_code,churn
2,apparel.belt,1.0
5,apparel.jeans,1.0
4,apparel.dress,1.0
12,apparel.tshirt,1.0
13,apparel.underwear,1.0
10,apparel.shoes.moccasins,1.0
7,apparel.shirt,1.0
36,appliances.kitchen.toster,1.0
35,appliances.kitchen.steam_cooker,1.0
16,appliances.environment.fan,1.0


#### 9.Churn por rango de precios

In [83]:
bins = [0, 50, 150, 300, 600, 1000, df["price"].max()]
labels = ["0-50", "50-150", "150-300", "300-600", "600-1000", "1000+"]

df["price_range"] = pd.cut(df["price"], bins=bins, labels=labels)

In [84]:
df_purchase = df[df["event_type"] == "purchase"].copy()

In [85]:
churn_price = (
    df_purchase
    .groupby(["price_range", "user_id"], observed=True)
    .size()
    .reset_index(name="purchase_count")
)

churn_price["churn"] = churn_price["purchase_count"] == 1

churn_price_rate = (
    churn_price
    .groupby("price_range", observed=True)["churn"]
    .mean()
    .reset_index()
    .sort_values(by="churn", ascending=False)
)

churn_price_rate

,price_range,churn
0,0-50,0.866260
1,50-150,0.856240
3,300-600,0.853351
2,150-300,0.852255
4,600-1000,0.842105
5,1000+,0.829677
